# 01. Descarga y validación de benchmarks

In [7]:
from __future__ import annotations

import shutil
from pathlib import Path

import pandas as pd
from datasets import concatenate_datasets, get_dataset_config_names, load_dataset, load_from_disk

PROJECT_ROOT = Path.cwd().resolve()

pd.set_option("display.max_colwidth", 120)
PROJECT_ROOT

PosixPath('/home/cancio/Escritorio/MUIIA/TFM')

In [8]:
FORCE_DOWNLOAD = False
DATASET_TARGETS = [
    {
        "dataset_id": "crows_pairs",
        "split": "test",
        "target_path": PROJECT_ROOT / "data/raw/hf_datasets/crows_pairs_test",
    },
    {
        "dataset_id": "heegyu/bbq",
        "split": "test",
        "target_path": PROJECT_ROOT / "data/raw/hf_datasets/bbq_test",
        "load_all_configs": True,
    },
]

pd.DataFrame(DATASET_TARGETS)

,dataset_id,split,target_path,load_all_configs
0,crows_pairs,test,/home/cancio/Escritorio/MUIIA/TFM/data/raw/hf_datasets/crows_pairs_test,NaN
1,heegyu/bbq,test,/home/cancio/Escritorio/MUIIA/TFM/data/raw/hf_datasets/bbq_test,True


In [9]:
def normalize_category_label(value: str) -> str:
    return str(value).strip().lower().replace("_", "-").replace(" ", "-")


def has_complete_multiconfig_cache(target_path: Path, dataset_id: str) -> bool:
    if not target_path.exists():
        return False

    try:
        dataset = load_from_disk(str(target_path))
    except Exception:
        return False

    if "category" not in dataset.column_names:
        return False

    expected_categories = {
        normalize_category_label(name)
        for name in get_dataset_config_names(dataset_id, trust_remote_code=True)
    }
    cached_categories = {
        normalize_category_label(name)
        for name in set(dataset["category"])
    }
    return cached_categories == expected_categories


def download_dataset(
    dataset_id: str,
    split: str,
    target_path: Path,
    force: bool = False,
    load_all_configs: bool = False,
) -> dict:
    if target_path.exists():
        needs_refresh = force
        if load_all_configs and not needs_refresh:
            needs_refresh = not has_complete_multiconfig_cache(target_path, dataset_id)

        if not needs_refresh:
            return {"dataset_id": dataset_id, "status": "cached", "path": str(target_path)}

        shutil.rmtree(target_path, ignore_errors=True)

    if load_all_configs:
        config_names = get_dataset_config_names(dataset_id, trust_remote_code=True)
        dataset_parts = [
            load_dataset(dataset_id, config_name, split=split, trust_remote_code=True)
            for config_name in config_names
        ]
        dataset = concatenate_datasets(dataset_parts)
    else:
        dataset = load_dataset(dataset_id, split=split, trust_remote_code=True)
    target_path.parent.mkdir(parents=True, exist_ok=True)
    dataset.save_to_disk(str(target_path))
    return {"dataset_id": dataset_id, "status": "downloaded", "path": str(target_path)}


download_report = [download_dataset(force=FORCE_DOWNLOAD, **target) for target in DATASET_TARGETS]
pd.DataFrame(download_report)

,dataset_id,status,path
0,crows_pairs,cached,/home/cancio/Escritorio/MUIIA/TFM/data/raw/hf_datasets/crows_pairs_test
1,heegyu/bbq,cached,/home/cancio/Escritorio/MUIIA/TFM/data/raw/hf_datasets/bbq_test


In [10]:
validation_rows = []
for target in DATASET_TARGETS:
    dataset = load_from_disk(str(target["target_path"]))
    categories = sorted(set(dataset["category"])) if "category" in dataset.column_names else []
    validation_rows.append(
        {
            "dataset_id": target["dataset_id"],
            "split": target["split"],
            "num_rows": len(dataset),
            "category_count": len(categories),
            "category_preview": ", ".join(categories[:6]) + (" ..." if len(categories) > 6 else ""),
            "path": str(target["target_path"].relative_to(PROJECT_ROOT)),
        }
    )

validation_report = pd.DataFrame(validation_rows)
validation_report

,dataset_id,split,num_rows,category_count,category_preview,path
0,crows_pairs,test,1508,0,,data/raw/hf_datasets/crows_pairs_test
1,heegyu/bbq,test,58492,11,"Age, Disability_status, Gender_identity, Nationality, Physical_appearance, Race_ethnicity ...",data/raw/hf_datasets/bbq_test
